# 05 — EEGNet (PyTorch CNN)

Trains the compact EEGNet architecture (Lawhern et al., 2018) directly on raw epoch tensors — no hand-crafted features required.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np
import torch

from models import EEGNet
from train import train_eegnet
from utils import set_seed

set_seed(42)
DATA_DIR = Path('../data')

X = np.load(DATA_DIR / 'X.npy')
y = np.load(DATA_DIR / 'y.npy')
print('X:', X.shape, 'y:', y.shape)
print('CUDA available:', torch.cuda.is_available())

## Inspect the architecture

In [ ]:
n_channels, n_times = X.shape[1], X.shape[2]
model = EEGNet(n_channels=n_channels, n_times=n_times, n_classes=len(np.unique(y)))
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {n_params:,}')

## Train with 5-fold cross-validation

In [ ]:
metrics = train_eegnet(X, y, n_epochs=80, lr=1e-3)
print(f"EEGNet accuracy: {metrics['accuracy_mean']:.3f} +/- {metrics['accuracy_std']:.3f}")
print(f"F1 (macro): {metrics['f1_macro_mean']:.3f}")
print(f"AUC-ROC: {metrics['auc_roc_mean']:.3f}")

## Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(metrics['history']['train_loss'], label='Train')
ax.plot(metrics['history']['val_loss'], label='Val', linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('EEGNet training curve (fold 1)')
ax.legend()
plt.show()

## Save metrics

In [ ]:
import json
metrics['model'] = 'EEGNet CNN'
with open('../results/eegnet_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved results/eegnet_metrics.json')